# Notebook 5 – Aggregation Features


## Learning objectives
- Understand aggregation features and their value in machine learning.
- Calculate count, sum, mean, median, minimum, maximum, and standard deviation.
- Create group-based, customer-level, product-level, and transaction-level features.
- Merge aggregated features into a modelling dataset.
- Avoid data leakage when aggregating historical data.

## 1. What are aggregation features?
Aggregation means summarizing multiple rows into meaningful values. An aggregation feature is a new feature created by applying a calculation to a group of records.

Example: if a customer makes five purchases, we can summarize the five transactions using total spending, average order value, purchase count, largest purchase, and variation in purchase amounts.

## 2. Why are aggregation features useful?
They convert detailed event records into compact behavioral summaries. They can reveal customer frequency, spending level, product popularity, and transaction patterns. These features may help models learn useful patterns for churn prediction, customer segmentation, demand forecasting, recommendation, and fraud detection.

A feature is useful only when it is relevant, correctly calculated, and available at prediction time.

## 3. Import libraries and create sample data
This notebook uses reproducible synthetic e-commerce data, so it runs without downloading a dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

transactions = pd.DataFrame({
    "transaction_id": [1001,1002,1003,1004,1005,1006,1007,1008,1009,1010,1011,1012],
    "customer_id": ["C01","C01","C01","C02","C02","C03","C03","C03","C04","C04","C05","C05"],
    "product_id": ["P01","P02","P01","P02","P03","P01","P03","P04","P02","P04","P01","P03"],
    "category": ["Electronics","Home","Electronics","Home","Beauty","Electronics","Beauty","Sports","Home","Sports","Electronics","Beauty"],
    "quantity": [1,2,1,1,3,2,1,1,2,1,1,2],
    "unit_price": [500,40,500,40,20,500,20,60,40,60,500,20],
    "rating": [5,4,5,3,4,5,4,5,3,4,5,4],
    "transaction_date": pd.to_datetime([
        "2026-01-02","2026-01-05","2026-02-10","2026-01-07",
        "2026-02-01","2026-01-12","2026-02-14","2026-03-01",
        "2026-01-20","2026-02-22","2026-03-05","2026-03-08"
    ])
})
transactions["transaction_amount"] = transactions["quantity"] * transactions["unit_price"]
display(transactions.head())

## 4. Aggregation functions

| Function | Meaning | Example |
|---|---|---|
| `count()` | Number of non-null values | Number of purchases |
| `sum()` | Total | Total spending |
| `mean()` | Arithmetic average | Average order value |
| `median()` | Middle value after sorting | Typical order amount |
| `min()` | Smallest value | Smallest purchase |
| `max()` | Largest value | Largest purchase |
| `std()` | Standard deviation / spread | Variation in purchase amounts |

**Mean vs median:** the mean can be pulled by extreme values. The median is often more robust when a group has unusually large or small transactions. Standard deviation measures spread, not average.

In [ ]:
amounts = pd.Series([100, 120, 130, 150, 1000])
print("Count:", amounts.count())
print("Sum:", amounts.sum())
print("Mean:", amounts.mean())
print("Median:", amounts.median())
print("Minimum:", amounts.min())
print("Maximum:", amounts.max())
print("Standard deviation:", amounts.std())

## 5. Group-based aggregation with `groupby()`
`groupby()` divides records into groups using one or more columns. We then calculate statistics for each group.

```python
data.groupby("group_column")["numeric_column"].sum()
```

In [ ]:
# Total spending by customer
display(transactions.groupby("customer_id")["transaction_amount"].sum())

# Multiple statistics for every customer
customer_summary = transactions.groupby("customer_id").agg(
    total_spending=("transaction_amount", "sum"),
    average_order_value=("transaction_amount", "mean"),
    median_order_value=("transaction_amount", "median"),
    number_of_purchases=("transaction_id", "nunique"),
    smallest_order=("transaction_amount", "min"),
    largest_order=("transaction_amount", "max"),
    spending_std=("transaction_amount", "std")
).reset_index()
display(customer_summary)

## 6. Customer-level aggregation
Customer-level features summarize all eligible purchases for each customer.

Examples:
- Total customer spending
- Average order value
- Number of purchases
- Maximum transaction amount
- Median transaction amount
- Spending standard deviation
- Total quantity purchased
- Number of unique products purchased

In [ ]:
customer_features = transactions.groupby("customer_id").agg(
    total_spending=("transaction_amount", "sum"),
    average_order_value=("transaction_amount", "mean"),
    number_of_purchases=("transaction_id", "nunique"),
    maximum_transaction=("transaction_amount", "max"),
    median_transaction=("transaction_amount", "median"),
    spending_variability=("transaction_amount", "std"),
    total_quantity=("quantity", "sum"),
    unique_products=("product_id", "nunique")
).reset_index()
display(customer_features)

### Interpreting customer features
- High total spending: higher revenue contribution.
- High purchase count: more frequent purchasing.
- High average order value: higher typical spend per order.
- Many unique products: broader product variety.
- High spending variability: less consistent purchase amounts.

These are possible interpretations; confirm them with business context. Standard deviation is `NaN` for a customer with only one observation because there is insufficient data to estimate variation.

## 7. Product-level aggregation
Product-level features summarize how customers interact with each product.

Examples: total units sold, total revenue, transaction count, average rating, minimum price, and maximum price.

In [ ]:
product_features = transactions.groupby("product_id").agg(
    total_units_sold=("quantity", "sum"),
    total_revenue=("transaction_amount", "sum"),
    transaction_count=("transaction_id", "nunique"),
    average_rating=("rating", "mean"),
    minimum_unit_price=("unit_price", "min"),
    maximum_unit_price=("unit_price", "max")
).reset_index()
display(product_features)

## 8. Transaction-level aggregation
If source data has multiple item rows per transaction, group those rows by transaction ID to create one summary row per transaction.

Possible features include transaction value, distinct products, total items, and average item price.

In [ ]:
transaction_features = transactions.groupby("transaction_id").agg(
    transaction_value=("transaction_amount", "sum"),
    distinct_products=("product_id", "nunique"),
    total_items=("quantity", "sum"),
    average_unit_price=("unit_price", "mean")
).reset_index()
display(transaction_features.head())

## 9. Group by multiple columns
You can group by multiple columns, such as category and product, to compare product performance within categories.

In [ ]:
category_product_summary = transactions.groupby(["category", "product_id"]).agg(
    total_revenue=("transaction_amount", "sum"),
    total_quantity=("quantity", "sum"),
    average_rating=("rating", "mean")
).reset_index()
display(category_product_summary)

## 10. Merge aggregated features into a modelling dataset
After calculating customer-level features, merge them into a customer profile table.

In [ ]:
customer_profile = pd.DataFrame({
    "customer_id": ["C01","C02","C03","C04","C05"],
    "age": [25,34,29,42,31],
    "city": ["Chennai","Salem","Coimbatore","Chennai","Madurai"]
})
modeling_data = customer_profile.merge(customer_features, on="customer_id", how="left")
display(modeling_data)

## 11. Time-aware aggregation and data leakage
**Data leakage** occurs when a feature uses information that would not have been available when the prediction is made.

Example: if the goal is to predict whether a customer will purchase next month, do not calculate their features using transactions from next month or later.

Define a prediction cutoff and aggregate only records before it.

In [ ]:
cutoff_date = pd.Timestamp("2026-03-01")
historical_transactions = transactions[
    transactions["transaction_date"] < cutoff_date
].copy()

customer_features_before_cutoff = historical_transactions.groupby("customer_id").agg(
    historical_spending=("transaction_amount", "sum"),
    historical_purchase_count=("transaction_id", "nunique"),
    historical_average_order_value=("transaction_amount", "mean")
).reset_index()
display(customer_features_before_cutoff)

### Leakage prevention checklist
- Define the prediction date or cutoff.
- Use only records available before that date.
- Exclude future transactions, future ratings, and future outcomes.
- For time-based predictions, consider rolling windows such as the previous 30, 90, or 365 days.
- Do not aggregate the target variable into features unless the design explicitly makes that valid.

## 12. Visualize customer spending

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(customer_features["customer_id"], customer_features["total_spending"])
plt.title("Total Spending by Customer")
plt.xlabel("Customer ID")
plt.ylabel("Total Spending")
plt.tight_layout()
plt.show()

## 13. Practical exercises
Try to solve these before running the solution cell:
1. Which customer has the highest total spending?
2. Which customer made the most purchases?
3. What is the overall average order value?
4. Which product has the highest revenue?
5. What is the average rating for each category?
6. Which customer has the highest spending variability?
7. Calculate total quantity purchased by each customer.
8. Recalculate customer features using transactions before `2026-02-01`.

In [ ]:
# Exercise solutions
print("1. Highest spending customer")
display(customer_features.nlargest(1, "total_spending"))

print("2. Customer with most purchases")
display(customer_features.nlargest(1, "number_of_purchases"))

print("3. Overall average order value")
print(transactions["transaction_amount"].mean())

print("4. Highest revenue product")
display(product_features.nlargest(1, "total_revenue"))

print("5. Average rating by category")
display(transactions.groupby("category")["rating"].mean().reset_index(name="average_rating"))

print("6. Highest spending variability")
display(customer_features.nlargest(1, "spending_variability"))

print("7. Total quantity by customer")
display(transactions.groupby("customer_id")["quantity"].sum().reset_index(name="total_quantity"))

print("8. Customer features before 2026-02-01")
early_data = transactions[transactions["transaction_date"] < pd.Timestamp("2026-02-01")]
display(early_data.groupby("customer_id").agg(
    spending=("transaction_amount", "sum"),
    purchase_count=("transaction_id", "nunique")
).reset_index())

## 14. Common mistakes
- Confusing transaction count with item quantity.
- Forgetting that `count()` counts non-null values in the selected column.
- Using mean when median better represents a skewed group.
- Ignoring missing values.
- Interpreting standard deviation for groups with too few records.
- Aggregating at the wrong level (customer, product, or transaction).
- Including future information and causing data leakage.
- Creating many features without checking relevance or redundancy.

## 15. Summary and conclusion
Aggregation features turn detailed records into compact summaries.

| Aggregation | Example feature |
|---|---|
| Count | Number of customer purchases |
| Sum | Total customer spending |
| Mean | Average order value |
| Median | Typical transaction amount |
| Minimum | Smallest transaction |
| Maximum | Largest transaction |
| Standard deviation | Variation in transaction amounts |
| Group-based aggregation | Sales by product or customer |

**Final takeaway:** Good aggregation features summarize relevant past behavior at the correct level and use only information available at prediction time. They can help machine learning models learn customer, product, and transaction patterns.